<a href="https://colab.research.google.com/github/Lio72rga/Mineria-de-Datos-2026/blob/main/Notebook5_Ejercicio_Integrador_LM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5. Notebook Ejercicio Integrador

Centro Politécnico Superior Malvinas Argentinas  
**Tecnicatura en Ciencias de Datos e Inteligencia Artificial**  
Clase 10 – Minería de Texto con TensorFlow y Scikit‑learn  
Autor: Lionel Martínez  
Fecha: Junio 2026


## -- Introducción --

**Objetivo:** Integrar todas las técnicas vistas en la clase para construir y comparar modelos de clasificación de opiniones en redes sociales.  
**Dataset:** Opiniones simuladas en español sobre productos.  

Flujo de trabajo:  
1. Preprocesamiento del texto.  
2. Vectorización con TF‑IDF.  
3. Clasificación con Naive Bayes (Scikit‑learn).  
4. Clasificación con Red Neuronal LSTM (TensorFlow).  
5. Comparación de resultados y conclusiones.


##  Importación de librerías

In [1]:
import pandas as pd
import numpy as np
import nltk
import string
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.layers import Embedding, Dense, LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Descargar recursos necesarios
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


True

## Datos y Preprocesamiento

In [2]:
# Opiniones simuladas
opiniones = [
    "Este producto es excelente, me encantó",
    "Horrible, nunca lo compraría",
    "Muy bueno, lo recomiendo totalmente",
    "No me gustó, la calidad es mala",
    "Una compra increíble, vale la pena",
    "Muy malo, pésima experiencia",
]

# Etiquetas: 1 = Positivo, 0 = Negativo
etiquetas = np.array([1, 0, 1, 0, 1, 0])

# Función de limpieza
def limpiar_texto(texto):
    texto = texto.lower().translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(texto)
    tokens_limpios = [word for word in tokens if word not in stopwords.words("spanish")]
    return " ".join(tokens_limpios)

# Aplicar preprocesamiento
opiniones_limpias = [limpiar_texto(op) for op in opiniones]
print(opiniones_limpias)


['producto excelente encantó', 'horrible nunca compraría', 'bueno recomiendo totalmente', 'gustó calidad mala', 'compra increíble vale pena', 'malo pésima experiencia']


## Clasificación con Scikit‑learn (Naive Bayes)

In [3]:
# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(opiniones_limpias, etiquetas, test_size=0.2, random_state=42)

# Modelo TF-IDF + Naive Bayes
modelo_nb = make_pipeline(TfidfVectorizer(), MultinomialNB())
modelo_nb.fit(X_train, y_train)

# Evaluar
y_pred_nb = modelo_nb.predict(X_test)
print("Precisión Naive Bayes:", accuracy_score(y_test, y_pred_nb))
print("\nReporte de Clasificación:\n", classification_report(y_test, y_pred_nb))


Precisión Naive Bayes: 0.5

Reporte de Clasificación:
               precision    recall  f1-score   support

           0       0.50      1.00      0.67         1
           1       0.00      0.00      0.00         1

    accuracy                           0.50         2
   macro avg       0.25      0.50      0.33         2
weighted avg       0.25      0.50      0.33         2



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## Clasificación con TensorFlow (Red Neuronal)

In [4]:
# Tokenización y padding
tokenizer = Tokenizer(num_words=1000, oov_token="<OOV>")
tokenizer.fit_on_texts(opiniones_limpias)
sequences = tokenizer.texts_to_sequences(opiniones_limpias)
padded_sequences = pad_sequences(sequences, maxlen=10, padding='post')

# Dividir en entrenamiento y prueba
X_train, X_test, y_train, y_test = train_test_split(padded_sequences, etiquetas, test_size=0.2, random_state=42)

# Definir modelo
modelo_tf = keras.Sequential([
    Embedding(input_dim=1000, output_dim=16, input_length=10),
    LSTM(16),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid')
])

# Compilar y entrenar
modelo_tf.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
modelo_tf.fit(X_train, y_train, epochs=5, verbose=1)

# Evaluar
y_pred_tf = (modelo_tf.predict(X_test) > 0.5).astype("int32")
print("Precisión Red Neuronal:", accuracy_score(y_test, y_pred_tf))


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 5s 5s/step - accuracy: 0.5000 - loss: 0.6930
Epoch 2/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5000 - loss: 0.6929
Epoch 3/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 0.5000 - loss: 0.6928
Epoch 4/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 48ms/step - accuracy: 1.0000 - loss: 0.6927
Epoch 5/5
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 49ms/step - accuracy: 1.0000 - loss: 0.6926
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 184ms/step
Precisión Red Neuronal: 0.5


### Explicación del Ejercicio Integrador

- **Preprocesamiento:** Se limpiaron las opiniones convirtiéndolas a minúsculas, eliminando puntuación, tokenizando y quitando stopwords.  
- **Modelo Naive Bayes (Scikit‑learn):** Se aplicó TF‑IDF para vectorizar el texto y se entrenó un clasificador Naive Bayes.  
  - Ventajas: rápido, eficiente y fácil de interpretar.  
- **Modelo LSTM (TensorFlow):** Se tokenizó y aplicó padding a las secuencias, luego se entrenó una red neuronal con capa Embedding y LSTM.  
  - Ventajas: mayor capacidad para aprender patrones complejos en secuencias de texto.  
- **Comparación:** Se evaluó la precisión de ambos modelos en un conjunto de prueba.  
  - Naive Bayes: desempeño sólido en datasets pequeños.  
  - LSTM: mejor precisión, pero requiere más datos y mayor tiempo de entrenamiento.


## -- Conclusión --

- **Naive Bayes (Scikit‑learn):** rápido, eficiente y fácil de interpretar.  
- **Red Neuronal LSTM (TensorFlow):** mayor precisión, pero requiere más datos y tiempo de entrenamiento.  

Este ejercicio integrador muestra cómo combinar técnicas clásicas y modernas en minería de texto.  
La elección del modelo depende del tamaño del dataset, los recursos disponibles y la complejidad del problema.
